### Execution of Tensor Train Decompositions on Grayscale Video
This notebook performs the core tensor decomposition experiments on the grayscale video sequence. It executes the proposed Online TT-ALS algorithm alongside three algebraic baselines: TT-FOA, Batch TT-ALS (Full), and Batch TT-ALS (Slice). The reconstructed tensors and computational times are saved locally to be evaluated by the subsequent metric calculation notebook.

In [ ]:
# ---------------------------------------------------------
# Environment Setup and Initialization
# ---------------------------------------------------------
using LinearAlgebra
using Random
using TensorToolbox
using Plots
using Images
using ImageIO
using FileIO
using FFMPEG_jll
using Printf
using Statistics
using Revise
using JLD2

# Explicitly restrict BLAS to a single thread.
# This ensures a strictly fair evaluation of the computational execution time 
# across all algorithms, as described in the experimental setup of the paper.
BLAS.set_num_threads(1)

In [ ]:
# Include custom modules containing the proposed method and baselines
includet("../julia_source/myGenerateData.jl")
includet("../julia_source/myLoadVideo.jl")
includet("../julia_source/myTTD.jl")

### Dataset Preparation & Loading (Grayscale Video)

Due to file size limits and licensing, the original video datasets are not included in this repository. 
To reproduce our experiments, please download the "office" sequence from the [CDnet 2014 benchmark](http://www.changedetection.net/).

**Note for custom video sequences:**
If you wish to evaluate the algorithms on your own custom grayscale video data, prepare a folder containing the sequential frame images and update the folder path in the `load_gray_video("path/to/your/custom/video")` function below.

In [ ]:
# ---------------------------------------------------------
# Dataset Loading and Preprocessing
# ---------------------------------------------------------
# Load the grayscale video sequence (e.g., CDnet 2014) which serves as the ground-truth streaming tensor.
video_tensor, h, w, T = load_gray_video("../CDNet/office/input")

([0.3607843137254902 0.3607843137254902 … 0.7372549019607844 0.7647058823529411; 0.3607843137254902 0.36470588235294116 … 0.7803921568627451 0.796078431372549; … ; 0.027450980392156862 0.027450980392156862 … 0.08627450980392157 0.0784313725490196; 0.027450980392156862 0.027450980392156862 … 0.08627450980392157 0.07450980392156863;;; 0.3607843137254902 0.3607843137254902 … 0.7333333333333333 0.7568627450980392; 0.3607843137254902 0.36470588235294116 … 0.7686274509803922 0.792156862745098; … ; 0.023529411764705882 0.023529411764705882 … 0.09019607843137255 0.09019607843137255; 0.023529411764705882 0.023529411764705882 … 0.10196078431372549 0.10196078431372549;;; 0.3686274509803922 0.37254901960784315 … 0.7411764705882353 0.7686274509803922; 0.37254901960784315 0.37254901960784315 … 0.7764705882352941 0.8; … ; 0.0196078431372549 0.0196078431372549 … 0.09411764705882353 0.09411764705882353; 0.01568627450980392 0.01568627450980392 … 0.09803921568627451 0.09411764705882353;;; … ;;; 0.3686274

In [19]:
# T = 1000;
# video_tensor = video_tensor[:, :, 1:T];
T = 500;
video_tensor = video_tensor[:, :, 351:350+T];
ttsizes = [h, w, T];
ttranks = [30, 30];

@show ttsizes;

ttsizes = [240, 360, 500]


In [5]:
@time save_gray_video(video_tensor, "results/gray", "office_original.mp4")

[ Info: Saved animation to /home/takeda/12/results/gray/office_original.mp4


109.521772 seconds (356.65 M allocations: 12.039 GiB, 0.76% gc time, 2.90% compilation time: 35% of which was recompilation)


In [ ]:
# ---------------------------------------------------------
# 1. Proposed Method: Online TT-ALS
# ---------------------------------------------------------

# Execute the exact, single-sweep online TT decomposition with incremental orthogonalization
G_history_est, gn_history_est, iter_time = online_ttALS(video_tensor, ttsizes, ttranks)

video_tensor_est = Array{Float64, 3}(undef, h, w, T)
relative_error = zeros(T)
for t in 1:T
    video_tensor_est[:, :, t] = ttProduct(G_history_est[t], gn_history_est[t])
    relative_error[t] = norm(video_tensor[:, :, t] - video_tensor_est[:, :, t]) / norm(video_tensor[:, :, t])
end
@printf("Average Relative Error: %.6f\n", mean(relative_error))

Average Relative Error: 0.067573


In [ ]:
# ---------------------------------------------------------
# 2. Baseline Method: TT-FOA (First-Order Approximation)
# ---------------------------------------------------------

# Execute the recursive online method based on first-order approximations.
# lambda: Forgetting factor for Recursive Least Squares (RLS).
# rho: Regularization parameter.
G_history_FOA_est, gn_history_FOA_est, iter_time_FOA = ttFOA(video_tensor, ttsizes, ttranks, 0.7, 1e-10)

video_tensor_FOA_est = Array{Float64, 3}(undef, h, w, T)
relative_error_FOA = zeros(T)
for t in 1:T
    video_tensor_FOA_est[:, :, t] = ttProduct(G_history_FOA_est[t], gn_history_FOA_est[t])
    relative_error_FOA[t] = norm(video_tensor[:, :, t] - video_tensor_FOA_est[:, :, t]) / norm(video_tensor[:, :, t])
end
@printf("Average Relative Error (FOA): %.6f\n", mean(relative_error_FOA))

Average Relative Error (FOA): 0.097530


In [ ]:
# ---------------------------------------------------------
# 3. Baseline Method: Batch TT-ALS (Full Sequence)
# ---------------------------------------------------------

# Apply standard TT-ALS directly to the entire 3D streaming tensor.
G_batch_est, iter_time_batch = @timed batch_ttALS(video_tensor, ttranks, max_sweeps=10, tol=1e-10)
iter_time_batch *=1e3  # in milliseconds
video_tensor_batch_est_full = tt2full(G_batch_est)
relative_error_batch_full = norm(video_tensor - video_tensor_batch_est_full) / norm(video_tensor)
@printf("Relative Error (Batch TT-ALS): %.6f\n", relative_error_batch_full)

Relative Error (Batch TT-ALS): 0.124061


In [ ]:
# ---------------------------------------------------------
# 4. Baseline Method: Batch TT-ALS (Slice-by-Slice)
# ---------------------------------------------------------

# Apply Batch TT-ALS sequentially to each 2D data slice independently.
iter_time_batchSlice = zeros(T)
G_batchSlice_est = Vector{Vector{Array{Float64}}}(undef, T)

for t in 1:T
    video_tensor_t = video_tensor[:, :, t]
    ttranks_t = [ttranks[1]]
    rlt = @timed batch_ttALS(video_tensor_t, ttranks_t, max_sweeps=10, tol=1e-10)
    G_batchSlice_est[t] = rlt.value
    iter_time_batchSlice[t] = rlt.time*1e3  # in milliseconds
end

video_tensor_batchSlice_est = Array{Float64, 3}(undef, h, w, T)
relative_error_batchSlice = zeros(T)
for t in 1:T
    video_tensor_batchSlice_est[:, :, t] = tt2full(G_batchSlice_est[t])
    relative_error_batchSlice[t] = norm(video_tensor[:, :, t] - video_tensor_batchSlice_est[:, :, t]) / norm(video_tensor[:, :, t])
end
@printf("Average Relative Error (Batch Slice): %.6f\n", mean(relative_error_batchSlice))

Average Relative Error (Batch Slice): 0.107507


In [ ]:
# ---------------------------------------------------------
# Save Experimental Results
# ---------------------------------------------------------
# Serialize the reconstructed video tensors and computational execution times to a JLD2 file.
# These intermediate results will be loaded by the evaluation notebook (ALS_indicator_gray.ipynb) 
# to independently compute the mathematical, structural, and perceptual quality metrics.

@save "data_gray_$(ttranks[1]).jld2" video_tensor_est video_tensor_FOA_est video_tensor_batch_est_full video_tensor_batchSlice_est iter_time iter_time_FOA iter_time_batch iter_time_batchSlice